# Example 8.2.2 / 1: Stop Early and Read What Came Back

Companion notebook for **8.2.2 Stopping Criteria and Reproducible Runs** in the AMPL lab Documentation, section "Stop Early and Read What Came Back". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Stop Early and Read What Came Back

This MIP chooses items to pack under a weight and a volume limit, the model used on
How MIP Solvers Work (AMPL lab documentation, section 8.9.1). The run sets a five-minute time limit, a one-percent relative
gap and `mip:return_gap=3`, which returns the absolute and relative gap; `mip:bestbound=1`
returns the bound. A second run adds `lim:improvingsols=1`, which stops at the first improving
solution. In `data.py` and `run.py`, `ampl` is an `AMPL` object that has already read
`model.mod`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                 # items that can be packed
param value {ITEMS} > 0;   # value of each item
param weight {ITEMS} > 0;  # weight of each item
param volume {ITEMS} > 0;  # volume of each item
param max_weight > 0;      # weight capacity
param max_volume > 0;      # volume capacity
var Take {ITEMS} binary;   # 1 if the item is packed
maximize Total_Value: sum {i in ITEMS} value[i] * Take[i];  # total value packed
# packed weight within capacity
subject to Weight: sum {i in ITEMS} weight[i] * Take[i] <= max_weight;
# packed volume within capacity
subject to Volume: sum {i in ITEMS} volume[i] * Take[i] <= max_volume;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
items = ["i1", "i2", "i3", "i4", "i5", "i6", "i7", "i8"]
ampl.set["ITEMS"] = items
ampl.param["value"] = dict(zip(items, [25, 47, 44, 18, 33, 48, 40, 50]))
ampl.param["weight"] = dict(zip(items, [23, 7, 24, 5, 20, 13, 22, 12]))
ampl.param["volume"] = dict(zip(items, [11, 27, 20, 22, 22, 20, 17, 25]))
ampl.param["max_weight"] = 63
ampl.param["max_volume"] = 82

In [5]:
# run.py
ampl.option["solver_msg"] = 0
base = "lim:time=300 mip:gap=0.01 mip:return_gap=3 mip:bestbound=1"
for opts in [base, base + " lim:improvingsols=1"]:
    ampl.solve(solver="highs", highs_options=opts)
    incumbent = ampl.obj["Total_Value"].value()
    bound = ampl.get_value("Total_Value.bestbound")
    gap = ampl.get_value("Total_Value.absmipgap")
    print(f"{ampl.solve_result}: incumbent {incumbent:g}, bound {bound:g}, gap {gap:g}")

HiGHS 1.14.0:   lim:time = 300
  mip:gap = 0.01
  mip:return_gap = 3
  mip:bestbound = 1

suffix absmipgap OUT;
suffix bestbound OUT;
suffix relmipgap OUT;
solved: incumbent 153, bound 153, gap 0
HiGHS 1.14.0:   lim:time = 300
  mip:gap = 0.01
  mip:return_gap = 3
  mip:bestbound = 1
  lim:improvingsols = 1
  Warning code 1 while solving with HiGHS
limit: incumbent 138, bound 178, gap 40


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. The `suffix` declarations let the commands
name the returned suffixes before the first solve:

In [6]:
%%writefile data.dat
param: ITEMS: value weight volume :=
  i1  25  23  11
  i2  47   7  27
  i3  44  24  20
  i4  18   5  22
  i5  33  20  22
  i6  48  13  20
  i7  40  22  17
  i8  50  12  25 ;
param max_weight := 63;
param max_volume := 82;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
suffix bestbound OUT;
suffix absmipgap OUT;
option highs_options 'lim:time=300 mip:gap=0.01 mip:return_gap=3 mip:bestbound=1';
solve;
display solve_result, Total_Value, Total_Value.bestbound, Total_Value.absmipgap;
option highs_options ($highs_options & ' lim:improvingsols=1');
solve;
display solve_result, Total_Value, Total_Value.bestbound, Total_Value.absmipgap;

HiGHS 1.14.0:   lim:time = 300
  mip:gap = 0.01
  mip:return_gap = 3
  mip:bestbound = 1

suffix relmipgap OUT;
solve_result = solved
Total_Value = 153
Total_Value.bestbound = 153
Total_Value.absmipgap = 0

HiGHS 1.14.0:   lim:time = 300
  mip:gap = 0.01
  mip:return_gap = 3
  mip:bestbound = 1
  lim:improvingsols = 1
  Warning code 1 while solving with HiGHS
solve_result = limit
Total_Value = 138
Total_Value.bestbound = 178
Total_Value.absmipgap = 40

